In [1]:
import requests
import pandas as pd
import os
from datetime import date


In [2]:
url = "https://portal.amfiindia.com/spages/NAVAll.txt"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
response.raise_for_status()


In [3]:
today = date.today().strftime("%Y%m%d")
raw_path = f"data/raw/nav_raw_{today}.txt"
os.makedirs("data/raw", exist_ok=True)

In [4]:
with open(raw_path, "w", encoding="utf-8") as f:
    f.write(response.text)


In [5]:
raw_lines = response.text.splitlines(keepends=True)
print(f"fetched and saved {len(raw_lines)} lines to {raw_path}")

fetched and saved 17974 lines to data/raw/nav_raw_20260829.txt


In [6]:
def is_data_row(line):
    parts = line.strip().split(";")
    return len(parts) >= 6 and parts[0].strip().isdigit()


In [7]:
def parse_row(line):
    parts = line.strip().split(";")
    scheme_code = parts[0].strip()
    isin_growth = parts[1].strip()
    isin_reinvest = parts[2].strip()
    nav = parts[-2].strip()
    nav_date = parts[-1].strip()
    scheme_name = " ".join(p.strip() for p in parts[3:-2] if p.strip())
    return [scheme_code, isin_growth, isin_reinvest, scheme_name, nav, nav_date]

In [8]:
def extract_rows_with_fund_house(raw_lines):
    current_fund_house = None
    rows = []

In [9]:
def extract_rows_with_fund_house(raw_lines):
    current_fund_house = None
    rows = []

    for line in raw_lines:
        stripped = line.strip()

        if not stripped:
            continue  # skip blank lines

        if stripped.startswith("Scheme Code"):
            continue  # column header line, not real data

        if is_data_row(line):
            row = parse_row(line)
            row.append(current_fund_house)  # tag with current fund house
            rows.append(row)
            continue

        # not a data row — either a category header or a fund house name
        is_category_line = stripped.startswith("Open Ended") or \
                            stripped.startswith("Close Ended") or \
                            stripped.startswith("Interval") or \
                            stripped.startswith("Exchange")

        if not is_category_line:
            current_fund_house = stripped

    return rows

In [10]:
rows = extract_rows_with_fund_house(raw_lines)
print(f"extracted {len(rows)} rows with fund house tagged")


extracted 14308 rows with fund house tagged


In [11]:
cols = ["scheme_code", "isin_growth", "isin_reinvest", "scheme_name", "nav", "nav_date", "fund_house"]
df = pd.DataFrame(rows, columns=cols)
df.head()

,scheme_code,isin_growth,isin_reinvest,scheme_name,nav,nav_date,fund_house
0,135762,INF846K01WO1,-,Axis Children's Fund Direct Plan Growth Option,30.5341,28-Aug-2026,Axis Mutual Fund
1,135765,INF846K01WP8,-,Axis Children's Fund Direct Plan IDCW Option,28.1274,28-Aug-2026,Axis Mutual Fund
2,135759,INF846K01WJ1,-,Axis Children's Fund Regular Plan Growth Option,26.6029,28-Aug-2026,Axis Mutual Fund
3,135760,INF846K01WK9,-,Axis Children's Fund Regular Plan IDCW Option,24.5433,28-Aug-2026,Axis Mutual Fund
4,135764,INF846K01WR4,-,Axis Children's Fund Direct Plan Growth Option,31.1180,28-Aug-2026,Axis Mutual Fund


In [12]:
print(f"distinct fund houses found: {df['fund_house'].nunique()}")
print(df["fund_house"].unique()[:10])
print(f"rows with missing fund house: {df['fund_house'].isna().sum()}")

distinct fund houses found: 52
['Axis Mutual Fund' 'Aditya Birla Sun Life Mutual Fund'
 'Franklin Templeton Mutual Fund' 'HDFC Mutual Fund' 'ITI Mutual Fund'
 'Kotak Mahindra Mutual Fund' 'Bandhan Mutual Fund'
 'Baroda BNP Paribas Mutual Fund' 'DSP Mutual Fund'
 'ICICI Prudential Mutual Fund']
rows with missing fund house: 0


In [13]:
df["nav"] = pd.to_numeric(df["nav"], errors="coerce")


In [14]:

# check how many rows broke during conversion, want to know before dropping them
bad_rows = df[df["nav"].isna()]
print(f"{len(bad_rows)} rows had a bad/missing nav value")
bad_rows.head()

0 rows had a bad/missing nav value


,scheme_code,isin_growth,isin_reinvest,scheme_name,nav,nav_date,fund_house


In [15]:
df["nav_date"] = pd.to_datetime(df["nav_date"], format="%d-%b-%Y", errors="coerce")

In [16]:
before = len(df)
df = df.dropna(subset=["nav", "nav_date"]).drop_duplicates()
print(f"{before} rows -> {len(df)} after dropping bad/dupe rows")


14308 rows -> 14308 after dropping bad/dupe rows


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14308 entries, 0 to 14307
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   scheme_code    14308 non-null  object        
 1   isin_growth    14308 non-null  object        
 2   isin_reinvest  14308 non-null  object        
 3   scheme_name    14308 non-null  object        
 4   nav            14308 non-null  float64       
 5   nav_date       14308 non-null  datetime64[ns]
 6   fund_house     14308 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 782.6+ KB


In [18]:
clean_path = f"data/clean/nav_clean_{today}.csv"
os.makedirs("data/clean", exist_ok=True)

In [19]:
df.to_csv(clean_path, index=False)
print(f"saved {len(df)} clean rows -> {clean_path}")

saved 14308 clean rows -> data/clean/nav_clean_20260829.csv
